# Generation of datasets to train MACE models
This notebook shows how to take a set of structures and use pycp2k and ASE to generate a dataset for training MACE models.
The dataset contains coordinates, energies, forces, and stress tensors for all the structures in the system, plus the isolated atoms and their energies.

## Import relevant modules
Apart from the usual suspects, we need to import the `mk_mace_dataset` function from the `pycp2k.workflows` module. This will allow us to generate a list of ASE Atoms objects, each with the energy, forces, and stress tensor calculated.

In [ ]:
from ase.io import write
import subprocess
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
from pycp2k.templates.FORCE_EVAL.PBE_templates import add_PBE_OT
from pycp2k.templates.PRINT.singlepoint import *
from pycp2k.workflows.mk_mace_dataset import load_dataset,get_elements,add_isolated_atoms

## Loading the dataset and adding the isolated atoms
We will use the `load_dataset` function to load the dataset from the `ds.xyz` file. This function will return a list of ASE Atoms objects, each with the charge and number of electrons calculated.

```python
from pycp2k.workflows.mk_mace_dataset import load_dataset

ds=load_dataset("ds.xyz")
```

We will also use the `get_elements` function to get the set of elements in the dataset, then add them as isolated atoms.

```python
from pycp2k.workflows.mk_mace_dataset import get_elements,add_isolated_atoms

symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
```

We can now print the first structures to check that the isolated atoms have been added.

In [ ]:
ds=load_dataset("ds.xyz")
symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
for system in ds[0:10]:
    print(system, system.info)

## Single point calculations and postprocessing
This is the main loop. For each structure in the data set (including the isolated atoms), we will run a single point calculation using CP2K, then we will extract energy, forces, and stress tensor and add it to the original ```Atoms``` object. Finally, we will write the data set to an ```extxyz``` file. The calculator can be changed between xTB and PBE just by commenting/uncommenting the relevant lines.

In [ ]:
nfailed=0
for system in ds[0:1]:
    print(system, system.info)
    calc=CP2K(project_name="mace_PBE",run_type="ENERGY_FORCE",cp2k_command="/home/joan/local/cp2k-master/bin/cp2k.psmp") # Need to redefine calculator every time?
    #add_xTB_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"],Ignore_convergence_failure=True,max_scf=1,outer_max_scf=0)
    add_PBE_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"],Ignore_convergence_failure=True,max_scf=1,outer_max_scf=0)
    forces_path=add_print_singlepoint_forces(calc=calc,filename="forces",unit=None)
    stress_path=add_print_stress_tensor(calc=calc,filename="./",unit=None)
    try:
        calc.run()
        system.info["E"]=postprocess_energy(calc=calc)
        system.set_array("forces",postprocess_forces(forces_path=forces_path))
        system.info["stress"]=postprocess_stress(stress_path=stress_path,notation="voigt")
    except Exception as e:
        print(f"Error: {e}")
        output_file=f"{calc.project_name}.out"
        subprocess.run(["tail", "-n", "30", output_file])
        nfailed+=1
        continue
    #finally:
        #calc.cleanup(quiet=True)

print(f"Number of failed calculations: {nfailed}")
#write("ds_ready.xyz",ds,format="extxyz")
    

In [ ]:
for system in ds:
    print(system)